## Destination standardization and port coordinate mapping
This stage is crucial for our analysis since without coordinates for each reported destination we cannot calculate haversine distance or estimate ETAs

In [ ]:
import pandas as pd
import numpy as np

df_target = pd.read_csv('../outputs/ais_eta_validated.csv')

During the data cleaning stage we saw that the top 200 most frequent destinations accounted for 73.6% of all observations. Based on our finding we will aprroach standardization in stages.

In [ ]:
print(df_target['Destination'].nunique())
dest_stats = df_target.groupby('Destination').agg(observations=('MMSI', 'size'), unique_vessels=('MMSI', 'nunique'))
dest_stats = dest_stats.sort_values('observations', ascending=False)
print(dest_stats.head(30))

In order to standardize and map destinations we'll need a reference dataset containing port codes. For this analysis we will use UN'S own UN/LOCODE dataset in csv form which contains LOCODES split into a 2-digit country code and a 3-digit location code e.x. SEGOT -> SE(Sweden) - GOT(Gotherburg) aswell as the approximate port coordinates. Unfortunately port coordinates are not in ordinary decimal latitide and longitude format which means we'll have to convert them to decimal degrees later to calculate haversine distance.

In [ ]:
part_1 = pd.read_csv('../data/reference/UNLOCODE CodeListPart1.csv', header=None)
part_2 = pd.read_csv('../data/reference/UNLOCODE CodeListPart2.csv', header=None)
part_3 = pd.read_csv('../data/reference/UNLOCODE CodeListPart3.csv', header=None)
print(part_1.shape, part_2.shape, part_3.shape)
port_codes = pd.concat([part_1, part_2, part_3], ignore_index=True)
columns = [
    'Change', 'Country', 'Location', 'Name',
    'NameWoDiacritics', 'SubDiv', 'Function',
    'Status', 'Date', 'IATA', 'Coordinates',
    'Remarks'
]
port_codes.columns = columns
print(port_codes.shape)
port_reference = port_codes[['Country', 'Location', 'Name', 'Function', 'Coordinates']].copy()
print(port_reference.shape)
port_reference['LOCODE'] = (port_reference['Country'].str.strip()) + (port_reference['Location'].str.strip())
print(port_reference['LOCODE'].count())
print(df_target['Destination'].isin(port_reference['LOCODE']).sum())
print(df_target['Destination'].count())

We can see that almost half of our observations already have a matching UN LOCODE. Keep in mind we havent handled port aliases and route-like destination strings.

In [ ]:
unique_dest = df_target['Destination'].drop_duplicates()
print(unique_dest.isin(port_reference['LOCODE']).sum())
unmatched = dest_stats[~dest_stats.index.isin(port_reference['LOCODE'])]
print(unmatched.head(50))

A total of 214 destination values match LOCODE directly leaving 412 unmatched. More importantly the ones that dont match are exactly the ones we expected (aliases and route-like/non-standard strings that need formating). Based on these findings we are going to build our destination mapping in layers.
- Layer 1: Exact LOCODE match  
Representing almost half of all observations and covering 214 out of the 626 unique destinations  

- Layer 2: Port aliases  
Match recognizable names and alternative spellings to LOCODES  
- Layer 3: Route-like strings  
Match the destination to LOCODE where the route is unambiguous  
- Layer 4: Unresolved destinations  
Leave these unmapped rather than invent ports and coordinates  
  
Before we wrap up layer 1 we need to check for any missing coordinates inside port_reference aswell as check for duplicate LOCODES

In [ ]:
print(port_reference['Coordinates'].isnull().sum())
print(port_reference['Coordinates'].notnull().sum())
valid_reference = port_reference['LOCODE'].dropna()
print(valid_reference.duplicated().sum())
duplicates = port_reference[['LOCODE', 'Coordinates']][(port_reference['LOCODE'].notnull()) & (port_reference['LOCODE'].duplicated(keep=False))]
print(duplicates.sort_values('LOCODE').tail(50))

We can see that there are 146 duplicate LOCODES. That doesnt really bother us unless any of them are inside our 214 matched group. Lets go ahead and check. If all of them are outside the exact matches we can leave them be as they wont be interfering with our analysis.

In [ ]:
print(unique_dest.isin(duplicates['LOCODE']).sum())
matched_duplicates = duplicates[duplicates['LOCODE'].isin(unique_dest)]
print(matched_duplicates.sort_values('LOCODE'))

As we can see there are only 9 duplicates out of the 214 matches. These mostly have identical coordinates with one having a pair of coordinates and a pair of NaN and another one having two empty pairs. Lets keep the identical ones with one record, the half pair with the existing pair and the empty one with no coordinates.

In [ ]:
port_reference = port_reference.sort_values('Coordinates', na_position='last')
port_reference = port_reference.drop_duplicates(subset='LOCODE', keep='first')
print(port_reference['LOCODE'].duplicated().sum())

Now we have no duplicates. Lets check for missing coordinates and then continue to merge the exact matches with the ais dataset

In [ ]:
available_codes = port_reference[(port_reference['LOCODE'].isin(unique_dest)) & (port_reference['Coordinates'].notnull())]
print(available_codes)
print(f"{(((df_target['Destination'].isin(available_codes['LOCODE']).sum()) / (df_target.shape[0])) * 100):.2f}%")
unavailable = port_reference[(port_reference['LOCODE'].isin(unique_dest)) & (port_reference['Coordinates'].isnull())]
print(dest_stats[dest_stats.index.isin(unavailable['LOCODE'])].sort_values('unique_vessels', ascending=False))
print(f"{(((dest_stats[dest_stats.index.isin(unavailable['LOCODE'])]['observations'].sum()) / df_target.shape[0]) * 100):.2f}%")

We can see that 150 ports match and have non-missing coordinates representing 32.15% of all observations in our validated dataset. The leftover 64 with missing coordinates take up 14.55% of observations. These 64 are absolutely worth fixing since coverage will jump to 46.65% which also lines up with the 4.55M exact match figure we saw earlier. We'll use a copy of WPI(World Port Index) from GitHub to check how many of those 64 ports we can cover using its LOCODE and Coordinate coverage

In [ ]:
wpi = pd.read_csv('../data/reference/world_port_index.csv')
cols = ['UN/LOCODE', 'Main Port Name', 'Latitude', 'Longitude']
wpi = wpi[cols]
print(wpi[wpi['UN/LOCODE'].isin(unavailable['LOCODE'])])
wpi_matches = wpi[wpi['Main Port Name'].isin(unavailable['Name'])]
print(wpi_matches)

After getting only 1 LOCODE match we tried Port Name matching instead to reveal any formatting difference. Turns out wpi uses a whitespace between the country and location so PLGDY becomes PL GDY. Lets remove the whitespace from wpis LOCODE column and try again

In [ ]:
wpi['UN/LOCODE'] = wpi['UN/LOCODE'].str.replace(' ', '')
print(unavailable['LOCODE'].isin(wpi['UN/LOCODE']).sum())
wpi_matches = wpi[wpi['UN/LOCODE'].isin(unavailable['LOCODE'])]
print(wpi_matches)
print(wpi_matches['UN/LOCODE'].count())
wpi_matches = wpi_matches[(wpi_matches['Main Port Name'] != 'Brunsbuttel Canal Terminals') & (wpi_matches['Main Port Name'] != 'Point Lisas Port')]
print(wpi_matches['UN/LOCODE'].count())

We have succesfully removed duplicates with the same LOCODE but different terminals of the same port so we've now gone from 60 to 57/64 ports. We're ready to merge the coordinates of wpi to unavailable.

In [ ]:
print(unavailable.shape[0])
print(unavailable['Coordinates'].isnull().sum())
wpi_coverage = pd.merge(unavailable, wpi_matches, how='left', left_on='LOCODE', right_on='UN/LOCODE')
print(wpi_coverage.shape[0])
print(wpi_coverage[['Latitude', 'Longitude']].notnull().sum())
print(wpi_matches['UN/LOCODE'].nunique())
print(wpi_coverage[wpi_coverage['Latitude'].isnull()])

Our next goal is to convert UNECES coordinate format to decimal degrees. Our UNECE dataset has coordinates placed inside a single column separated by whitespace while wpi has individual lat and lon columns. We also have to think about how to extract degrees, minutes and direction from the coordinate strings. We'll apply all this to a single destination and confirm it works then refactor it to a reuseable function we can apply to port_reference

In [ ]:
print(port_reference[port_reference['LOCODE'] == 'NOTOS'])
cor_reference = port_reference[port_reference['LOCODE'] == 'NOTOS'].copy()


def to_decimal(coordinate: str):
    raw = coordinate
    if pd.isnull(raw) :
        return np.nan
    elif len(raw) == 5:
        lat_deg = float(raw[:2])
        lat_min = int(raw[2:4])
        lat_dir = raw[4:5]
        return (-(lat_deg + (lat_min/60)) if lat_dir == 'S' else lat_deg + (lat_min/60))
    elif len(raw) == 6:
        lon_deg = float(raw[:3])
        lon_min = int(raw[3:5])
        lon_dir = raw[5:6]
        return (-(lon_deg + (lon_min/60)) if lon_dir == 'W' else lon_deg + (lon_min/60))
    else:
        return np.nan

print(to_decimal('6940N'))
print(to_decimal('0803S'))
print(to_decimal('01858E'))
print(to_decimal('06001W'))
print(to_decimal(np.nan))

Since our function works as intended we are ready to apply it to our desired dataset 

In [ ]:
port_lat = (port_reference['Coordinates'].str.split()).str[0]
port_lon = (port_reference['Coordinates'].str.split()).str[1]
port_reference['Latitude'] = port_lat.apply(to_decimal)
port_reference['Longitude'] = port_lon.apply(to_decimal)
print(port_reference[port_reference['LOCODE'] == 'NOTOS'])
print(port_reference[['Latitude', 'Longitude']].notnull().sum())
print(port_reference[['Latitude', 'Longitude']].count())
print(port_reference['Latitude'][port_reference['Latitude'] > 90].count())
print(port_reference['Latitude'][port_reference['Latitude'] < -90].count())
print(port_reference['Longitude'][port_reference['Longitude'] > 180].count())
print(port_reference['Longitude'][port_reference['Longitude'] < -180].count())
print(port_reference['Latitude'][(port_reference['Latitude'] == 0) & (port_reference['Longitude'] == 0)].count())

Our function has successfully converted all 93,050 Coordinates to valid non-null decimal degrees. All latitude values are valid  however we have produced 3 longitude values over 180 degrees deeming them invalid. We have also produced 2 coordinate pairs with latitude  Lets investigate them.

In [ ]:
print(port_reference[port_reference['Longitude'] > 180])
malformed_lon = port_reference[port_reference['Longitude'] > 180]
print(malformed_lon['LOCODE'].isin(unique_dest).sum())
point_null = port_reference[(port_reference['Latitude'] == 0) & (port_reference['Longitude'] == 0)]
print(point_null['LOCODE'].isin(unique_dest).sum())

None of these values are from ports that interest us so we can just go ahead and leave them there since they wont bother us. Since we now have valid coordinate pairs on both our UNECE and wpi datasets we can combine them

In [ ]:
wpi_reference = wpi_coverage[['LOCODE', 'Latitude', 'Longitude']][(wpi_coverage['Latitude'].notnull()) & (wpi_coverage['Longitude'].notnull())]
print(wpi_reference.shape)
print(port_reference.shape)
enriched_reference = pd.merge(port_reference, wpi_reference, how='left', on='LOCODE')
print(enriched_reference.shape)
print(enriched_reference[enriched_reference['LOCODE'] == 'PLGDN'])

We have successfuly merged the two datasets into an enriched port lookup dataset. It has the same number of rows as the original port reference but has two additional columns which are wpis latitude and longitude columns. Since both datasets had latitude and longitude columns our enriched dataset has 2 sets of coordinate pairs: _x and _y. _x is UNECES converted latitude and longitude pair while _y is wpis coordinate pair. We'll merge these columns prioritizing UNECE where available and using wpi coordinates to fill missing values like lat_x and lon_x in PLGDN. We'll also add a source column so its easy to trace where each coordinate came from.

In [ ]:
enriched_reference['Latitude'] = enriched_reference['Latitude_x'].fillna(enriched_reference['Latitude_y'])
enriched_reference['Longitude'] = enriched_reference['Longitude_x'].fillna(enriched_reference['Longitude_y'])
enriched_reference['Source'] = 'Unresolved'
enriched_reference.loc[(enriched_reference['Latitude_x'].notnull()) & (enriched_reference['Longitude_x'].notnull()), 'Source'] = 'UNECE'
enriched_reference.loc[(enriched_reference['Latitude_y'].notnull()) & (enriched_reference['Longitude_y'].notnull()) & (enriched_reference['Source'] != 'UNECE'), 'Source'] = 'WPI'
print(enriched_reference['Source'].value_counts())
print(enriched_reference['Source'][enriched_reference['LOCODE'] == 'NOTOS'])
print(enriched_reference['Source'][enriched_reference['LOCODE'] == 'PLGDN'])
dropped = ['Latitude_x', 'Latitude_y', 'Longitude_x', 'Longitude_y']
enriched_reference = enriched_reference.drop(columns=dropped)

Our enriched port reference dataset now has a latitude and longitude column aswell as a source column all working as intended. Lets check the updated destination coverage after resolving 57 of the 64 unavailable coordinate pairs 

In [ ]:
updated_dest_coverage = enriched_reference[(enriched_reference['LOCODE'].isin(unique_dest)) & (enriched_reference['Latitude'].notnull()) & (enriched_reference['Longitude'].notnull())]
print(updated_dest_coverage.count())
print(f"{((df_target['Destination'].isin(updated_dest_coverage['LOCODE']).sum() / df_target.shape[0]) * 100):.2f}%")
print(df_target['Destination'].isin(updated_dest_coverage['LOCODE']).sum() - df_target['Destination'].isin(available_codes['LOCODE']).sum())

We have substantially increased our geographic coverage from 32.14% to 46.14% covering an additional 1,364,649 observations. There are still 7 left over destinations with no coordinates from the total 64 original unmatched ones. Lets see how much coverage these 7 provide before deciding wether we drop them or not.

In [ ]:
unavailable = enriched_reference[(enriched_reference['LOCODE'].isin(unique_dest)) & (enriched_reference['Latitude'].isnull()) & (enriched_reference['Longitude'].isnull())]
left_over = dest_stats[['observations', 'unique_vessels']][dest_stats.index.isin(unavailable['LOCODE'])]
print(left_over)
print(enriched_reference[enriched_reference['LOCODE'] == 'DETRV'])
print(enriched_reference[enriched_reference['LOCODE'] == 'DETRA'])

Manually mapping left over destination coordinates would increase our coverage by 54,395 observations and 11 unique vessels. Its safe to say that a conservative manual map is a pretty good choice for these 7 destinations. Lets start with the first 2 which have a significant number of observations in comparison to the rest. Turns out DETRA is a legacy port code variant frequently maintained in proprietary logistics databases, freight forwarding systems and historical port tables. In the official UN/LOCODE registry, DETRA is actually assigned to the inland town of Traunstein in southern Germany meaning the actual port of DETRA is travemunde(DETRV). We can safely map DETRA to DETRVS coordinates. We will manually map the rest of the destination coordinates. For the sake of traceability we will set their source to 'Manual' 

In [ ]:
enriched_reference.loc[enriched_reference['LOCODE'] == 'DETRA', 'Latitude'] =  53.966667   
enriched_reference.loc[enriched_reference['LOCODE'] == 'DETRA', 'Longitude'] = 10.883333
enriched_reference.loc[enriched_reference['LOCODE'] == 'DETRA', 'Source'] = 'Manual'

enriched_reference.loc[enriched_reference['LOCODE'] == 'GBRCS', 'Latitude'] = 51.390495
enriched_reference.loc[enriched_reference['LOCODE'] == 'GBRCS', 'Longitude'] = 0.5182773
enriched_reference.loc[enriched_reference['LOCODE'] == 'GBRCS', 'Source'] = 'Manual'

enriched_reference.loc[enriched_reference['LOCODE'] == 'DKLIN', 'Latitude'] = 55.4666
enriched_reference.loc[enriched_reference['LOCODE'] == 'DKLIN', 'Longitude'] = 10.5333
enriched_reference.loc[enriched_reference['LOCODE'] == 'DKLIN', 'Source'] = 'Manual'

enriched_reference.loc[enriched_reference['LOCODE'] == 'FIVKO', 'Latitude'] = 60.4039
enriched_reference.loc[enriched_reference['LOCODE'] == 'FIVKO', 'Longitude'] = 26.2556
enriched_reference.loc[enriched_reference['LOCODE'] == 'FIVKO', 'Source'] = 'Manual'

enriched_reference.loc[enriched_reference['LOCODE'] == 'DEMUK', 'Latitude'] = 54.4666
enriched_reference.loc[enriched_reference['LOCODE'] == 'DEMUK', 'Longitude'] = 13.5833
enriched_reference.loc[enriched_reference['LOCODE'] == 'DEMUK', 'Source'] = 'Manual'

enriched_reference.loc[enriched_reference['LOCODE'] == 'DEHBG', 'Latitude'] = 53.507
enriched_reference.loc[enriched_reference['LOCODE'] == 'DEHBG', 'Longitude'] = 9.9619
enriched_reference.loc[enriched_reference['LOCODE'] == 'DEHBG', 'Source'] = 'Manual'

enriched_reference.loc[enriched_reference['LOCODE'] == 'NOKVD', 'Latitude'] = 58.28333333333
enriched_reference.loc[enriched_reference['LOCODE'] == 'NOKVD', 'Longitude'] = 6.966666666667
enriched_reference.loc[enriched_reference['LOCODE'] == 'NOKVD', 'Source'] = 'Manual'

print(enriched_reference[(enriched_reference['LOCODE'].isin(unique_dest)) & (enriched_reference['Latitude'].isnull()) & (enriched_reference['Longitude'].isnull())])
updated_dest_coverage = enriched_reference[(enriched_reference['LOCODE'].isin(unique_dest)) & (enriched_reference['Latitude'].notnull()) & (enriched_reference['Longitude'].notnull())]
print(f"{((df_target['Destination'].isin(updated_dest_coverage['LOCODE']).sum() / df_target.shape[0]) * 100):.2f}%")
print(df_target['Destination'].isin(updated_dest_coverage['LOCODE']).sum() - df_target['Destination'].isin(available_codes['LOCODE']).sum())

This marks layer 1 of our destination mapping as finished. Our next goal is layer 2: Port aliases where we will match recognizable names and alternative spellings to individual LOCODES.

In [ ]:
unmatched_dests = dest_stats[~dest_stats.index.isin(enriched_reference['LOCODE'])]
layer_2 = df_target[(df_target['Destination'].isin(unmatched_dests.index)) & (~df_target['Destination'].str.contains(r'[<>=/-]',na=False))]
print(layer_2['Destination'].nunique())
layer_2_stats = layer_2.groupby('Destination').agg(observations=('MMSI', 'size'), unique_vessels=('MMSI', 'nunique'))
print(layer_2_stats.sort_values('observations', ascending=False).head(50))

We have 267 unique destinations without route-like string characters like > and ->. Not all 267 of them are valid layer 2 destinations since we have destinations like SKAGENFORORDER  and FISHINGGROUND. We'll make a new dataframe and a mapping dictionary manually mapping recognizable port names, spelling variants, and legacy/alternative names to LOCODEs, then apply those mappings to all observations.

In [ ]:
layer_2_mapping = layer_2_stats.copy()
layer_2_mapping['class'] = 'Unresolved'
layer_2_mapping['mapped_LOCODE'] = 'Unresolved'
mapping_dict = {
    'SKAGEN' : 'DKSKA',
    'MUNDRA' : 'INMUN',
    'AARHUS' : 'DKAAR',
    'ROSTOCK' : 'DERSK',
    'USTLUGA' : 'RUULU',
    'DEHAMCTB' : 'DEHAM',
    'GOTEBORG' : 'SEGOT',
    'ESBJERG' : 'DKEBJ',
    'MALMO' : 'SEMMA',
    '0RAN.ALGERIA' : 'DZORN',
    'HELSINKI' : 'FIHEL',
    'HANKO' : 'FIHKO',
    'KLAIPEDA' : 'LTKLJ',
    'GOTHENBURG' : 'SEGOT',
    'PRIMOSK' : 'RUPRI',
    'NAKSKOV' : 'DKNAK',
    'TRELLEBORG' : 'SETRG',
    'RUDKOEBING' : 'DKRKB',
    'HELSINGBORG' : 'SEHEL',
    'FREDERICIA' : 'DKFRC',
    'FALKENBERG' : 'SEFAG',
    'PORTSAID,EGYPT' : 'EGPSD',
    'KRAGENAES' : 'DKKRA',
    'MALMOE' : 'SEMMA',
    'GDYNIA' : 'PLGDY',
    'PASAJES' : 'ESPAS',
    'LINDO' : 'DKLIN',
    'MOTRIL' : 'ESMOT',
    'SETRE' : 'SETRG',
    'WISMAR' : 'DEWIS',
    'VITORIA' : 'BRVIX',
    'WOLGAST' : 'DEWOL',
    'DELFZIJL' : 'NLDZL',
    'PRAIAMOLE' : 'BRPRM',
    'STIGSNAES' : 'DKSTN',
    'BANDHOLM' : 'DKBDX',
    'SEJEROE' : 'DKSEO',
    'FIHELSINKIVUOSAARI' : 'FIVSS'
}
print(enriched_reference['LOCODE'].isin(mapping_dict.values()).sum())
missing = set(mapping_dict.values()) - set(enriched_reference['LOCODE'])
print(missing)
recovered = df_target[df_target['Destination'].isin(mapping_dict.keys())]
print(recovered['MMSI'].nunique())
print(recovered['Destination'].count())
print(recovered['Destination'].nunique())

We've mapped 38 high confidence port-aliases to LOCODES. All 38 of them exist inside enriched reference. This first batch of the most observation dense destinations has given us 1,248,856 brand new observations aswell as 67 unique vessels. We will map these to layer_2_mapping and continue with a second batch.

In [ ]:
layer_2_mapping.loc[layer_2_mapping.index.isin(mapping_dict.keys()), 'mapped_LOCODE'] = layer_2_mapping.index.to_series().map(mapping_dict)
layer_2_mapping.loc[layer_2_mapping.index.isin(mapping_dict.keys()), 'class'] = 'port_alias'
print((layer_2_mapping['mapped_LOCODE'] == 'Unresolved').sum())
print((layer_2_mapping['class'] == 'port_alias').sum())
print(layer_2_mapping[layer_2_mapping['mapped_LOCODE'] == 'SEGOT'])

Our mapping is working as intended. Lets continue to batch 2.

In [ ]:
batch_2_stats = layer_2_mapping[(layer_2_mapping['mapped_LOCODE'] == 'Unresolved') & (layer_2_mapping['class'] == 'Unresolved')]
print(batch_2_stats.sort_values('observations', ascending=False).head(50))

mapping_dict2 = {
    'DEKIEL': 'DEKEL',
    'COPENHAGEN': 'DKCPH',
    'GIBRALTAR': 'GIGIB',
    'TROLLHATTAN': 'SETHN',
    'FREDERICA': 'DKFRC',
    'TEMA': 'GHTEM',
    'KALUNDBORG': 'DKKAL',
    'SANDNES': 'NOSAS',
    'ZEEBRUGGE': 'BEZEE',
    'SUEZ': 'EGSUZ',
    'SASSNITZ': 'DESAS',
    'KARLSKRONA': 'SEKAA',
    'REKEFJORD': 'NOREK',
    'MANDAL': 'NOMAN',
    'HIRTSHALS': 'DKHIR',
    'SZCZECIN': 'PLSZZ',
    'MARIAGER': 'DKMRR',
    'AALBORG': 'DKAAL',
    'AABENRAA': 'DKAAB',
    'NEWORLEAN': 'USMSY',
    'KBH': 'DKCPH',
    'TERNEUZEN': 'NLTNZ',
    'TILBURY': 'GBTIL',
    'HAMINA': 'FIHMN',
    'RONNE': 'DKRNN'
}

operational_dict = {
    'SKAGENFORORDER': 'operational_text',
    'EETLLORDERS': 'operational_text',
    'TALLINFORORDERS': 'operational_text',
    'TALLINNF.O.': 'operational_text',
    'LIBYAFORORDER': 'operational_text',
    'GIBRALTARFORORDERS': 'operational_text',
    'BALTICSEAFORORDER': 'operational_text'
}

print(enriched_reference['LOCODE'].isin(mapping_dict2.values()).sum())
print(set(mapping_dict2.values()) - set(enriched_reference['LOCODE']))
recovered2 = df_target[df_target['Destination'].isin(mapping_dict2.keys())]
print(recovered2['MMSI'].nunique())
print(recovered2['Destination'].count())
print(recovered2['Destination'].nunique())

layer_2_mapping.loc[layer_2_mapping.index.isin(mapping_dict2.keys()), 'mapped_LOCODE'] = layer_2_mapping.index.to_series().map(mapping_dict2)
layer_2_mapping.loc[layer_2_mapping.index.isin(mapping_dict2.keys()), 'class'] = 'port_alias'
layer_2_mapping.loc[layer_2_mapping.index.isin(operational_dict.keys()), 'class'] = layer_2_mapping.index.to_series().map(operational_dict)
print(layer_2_mapping[layer_2_mapping['mapped_LOCODE'] == 'DKCPH'])

With batch 2 finished we've recovered another 364,121 observations and 38 unique vessels. Our layer 2 total so far is 1,612,977 observations and 105 unique vessels. We'll continue with the third batch taking rows with their class column being 'Unresolved'.

In [ ]:
batch_3_stats = layer_2_mapping[(layer_2_mapping['mapped_LOCODE'] == 'Unresolved') & (layer_2_mapping['class'] == 'Unresolved')]
print(batch_3_stats.sort_values('observations',ascending=False).head(60))

mapping_dict3 = {
    'JOSSINGFJORD': 'NOJOS',
    'HEILIGENHAFEN': 'DEHHF',
    'KOLOBRZEG': 'PLKOL',
    'LUBECK': 'DELBC',
    'KOGE': 'DKKOG',
    'SARPSBORG': 'NOSPG',
    'BREMERHAVEN': 'DEBRV',
    'SKAW': 'DKSKA',
    'LYSEKIL': 'SELYS',
    'UDDEVALLA': 'SEUDD',
    'SANKTPETERBURG': 'RULED',
    'NYKOBINGFALSTER': 'DKNYF',
    'KIEL': 'DEKEL',
    'HADERSLEV': 'DKHAD',
    'STORUGNS': 'SESUS',
    'GOTHENBORG': 'SEGOT',
    'DRAMMEN': 'NODRM',
    'VASTERAS': 'SEVST',
    'MUKRAN': 'DEMUK',
    'EGERNSUND': 'DKEND',
    'POLICE': 'PLPLC',
    'DARLOWO': 'PLDAR',
    'BERGEN': 'NOBGO',
    'SKAW,DENMARK': 'DKSKA',
    'TALLINN': 'EETLL',
    'SLITE': 'SESLI',
    'WARNEMUNDE,DE': 'DEWAR',
    'GOTHENBURGPEC63' : 'SEGOT',
    'DKCPHPEC210' : 'DKCPH',
    'DEHAMVIANOK': 'DEHAM',
    'GDANSKVIASKAW': 'PLGDN',
    'GHENTVIANOK': 'BEGNE',
    'SEVBYVIAKLMR': 'SEVBY',
    'AMSTERDAMVIANOK': 'NLAMS',
    'LVRIXVIANOK': 'LVRIX',
    'DERENVIANOK': 'DEREN'
}

operational_dict3 = {
    'FERRYDKSE': 'operational_text',
    'ORDER': 'operational_text',
    'BALTICFORORDER': 'operational_text'
}

print(enriched_reference['LOCODE'].isin(mapping_dict3.values()).sum())
print(set(mapping_dict3.values()) - set(enriched_reference['LOCODE']))
recovered3 = df_target[df_target['Destination'].isin(mapping_dict3.keys())]
print(recovered3['MMSI'].nunique())
print(recovered3['Destination'].count())
print(recovered3['Destination'].nunique())

layer_2_mapping.loc[layer_2_mapping.index.isin(mapping_dict3.keys()), 'mapped_LOCODE'] = layer_2_mapping.index.to_series().map(mapping_dict3)
layer_2_mapping.loc[layer_2_mapping.index.isin(mapping_dict3.keys()), 'class'] = 'port_alias'
layer_2_mapping.loc[layer_2_mapping.index.isin(operational_dict3.keys()), 'class'] = layer_2_mapping.index.to_series().map(operational_dict3)

Batch 3 has brought us another 380,300 new observations and 48 unique vessels totaling 1,993,277 observations and 153 vessels. We'll do one final batch because we've established that batch returns are diminishing.

In [ ]:
batch_4_stats = layer_2_mapping[(layer_2_mapping['mapped_LOCODE'] == 'Unresolved') & (layer_2_mapping['class'] == 'Unresolved')]
#print(batch_4_stats.sort_values('observations',ascending=False).head(60))

mapping_dict4 = {
    'HARDING': 'GBHRG',
    'GENK': 'BEGNK',
    'DELBC[LUBECK]': 'DELBC',
    'BROFJORDEN': 'SEBRO',
    'FOKLAKSVIK': 'FOKVI',
    'SONDERBORG': 'DKSGD',
    'FREDERIKSHAVN': 'DKFDH',
    'ASSENS': 'DKASN',
    'PORVOO': 'FIPRV',
    'AHUS': 'SEAHU',
    'HULLVIANOK': 'GBHUL',
    'PLGDYVIANOK': 'PLGDY',
    'PLGDNVIANOK': 'PLGDN',
    'DKKALPEC473': 'DKKAL',
    'GDANSK': 'PLGDN',
    'DKGRE(GRENAA)': 'DKGRE',
    'HULL': 'GBHUL',
    'TORNIO': 'FITOR',
    'KRISTIANSAND': 'NOKRS',
    'GOTHENBURGSE': 'SEGOT',
    'DKANH(ANHOLT)': 'DKANH',
    'BERGEN,NORWAY': 'NOBGO',
    'ROTTERDAM': 'NLRTM',
    'VIEROW': 'DEVIW',
    'ZINGST': 'DEZIS',
    'JORFLASFAR': 'MAJFL'
}

operational_dict4 = {
    'GULFOFFINLAND': 'operational_text',
    'FISHINGGROUND': 'operational_text',
    'BALTICSEA': 'operational_text',
    'BALTIC2OWF': 'operational_text',
    'BALTICPOWEROWF': 'operational_text',
    'HORNSREV3': 'operational_text',
    'WIKINGERWF': 'operational_text',
    'YME': 'operational_text',
    'XZNORNE': 'operational_text',
    'FERRYSERVICE': 'operational_text'
}

print(enriched_reference['LOCODE'].isin(mapping_dict4.values()).sum())
print(set(mapping_dict4.values()) - set(enriched_reference['LOCODE']))
recovered4 = df_target[df_target['Destination'].isin(mapping_dict4.keys())]
print(recovered4['MMSI'].nunique())
print(recovered4['Destination'].count())
print(recovered4['Destination'].nunique())

layer_2_mapping.loc[layer_2_mapping.index.isin(mapping_dict4.keys()), 'mapped_LOCODE'] = layer_2_mapping.index.to_series().map(mapping_dict4)
layer_2_mapping.loc[layer_2_mapping.index.isin(mapping_dict4.keys()), 'class'] = 'port_alias'
layer_2_mapping.loc[layer_2_mapping.index.isin(operational_dict4.keys()), 'class'] = layer_2_mapping.index.to_series().map(operational_dict4)

total_recovered = recovered['Destination'].count() + recovered2['Destination'].count() + recovered3['Destination'].count() + recovered4['Destination'].count()
print(f"{((total_recovered / df_target.shape[0]) * 100):.2f}%")
print(layer_2_mapping[layer_2_mapping['class'] == 'port_alias'].count())
print(layer_2_mapping[layer_2_mapping['class'] == 'operational_text'].count())
print(layer_2_mapping[layer_2_mapping['class'] == 'Unresolved'].count())

We have successfully finished batch 4 bringing in 165,334 observations and 28 more vessels. The final layer 2 total is 2,158,611 observations covering 22.14% of all observations and our 2 layer summary being 68.84%.

In [ ]:
layer_2_reference = layer_2_mapping[(layer_2_mapping['mapped_LOCODE'] != 'Unresolved') & (layer_2_mapping['class'] == 'port_alias') & (layer_2_mapping['mapped_LOCODE'].isin(enriched_reference['LOCODE']))]
layer_2_reference = layer_2_reference[['observations', 'unique_vessels', 'class', 'mapped_LOCODE']]
print(layer_2_reference.shape[0])
print(layer_2_reference['mapped_LOCODE'].isnull().sum())

The final step before concluding layer 2 would be to check if any of the destinations inside layer_2_reference dont have coordinates in our enriched dataset

In [ ]:
enriched_nnull = enriched_reference[enriched_reference['Latitude'].notnull() & enriched_reference['Longitude'].notnull()]
print(layer_2_reference['mapped_LOCODE'].isin(enriched_nnull['LOCODE']).sum())
print(layer_2_reference['mapped_LOCODE'].count())
missing_layer2 = set(layer_2_reference['mapped_LOCODE']) - set(enriched_nnull['LOCODE'])
print(missing_layer2)
print(layer_2_reference[layer_2_reference['mapped_LOCODE'].isin(missing_layer2)])
print(layer_2_reference['observations'][layer_2_reference['mapped_LOCODE'].isin(missing_layer2)].sum())
print(layer_2_reference['unique_vessels'][layer_2_reference['mapped_LOCODE'].isin(missing_layer2)].sum())

22 destinations in layer 2 dont have coordinates. They cover a substantial amount of observations (432,874) aswell as 28 unique vessels.Throwing them away would not be wise. Instead we will find reliable coordinates for each. We will make another mapping dictionary to map the new coordinates to enriched reference.

In [ ]:
manual_coords = {
    'ESMOT': (36.7193200, -3.5220860),   
    'DKSEO': (55.8807000, 11.1339000),   
    'DZORN': (35.7166667, -0.6500000),   
    'GIGIB': (36.1438000, -5.3647500),   
    'ESPAS': (43.3240000, -1.9177000),   
    'DKNAK': (54.8333333, 11.1333333),   
    'DKHAD': (55.2503906, 9.5048067),   
    'MAJFL': (33.1272222, -8.6277778),  
    'DKRKB': (54.9389400, 10.7050500), 
    'DESAS': (54.5105000, 13.6395000),   
    'FOKVI': (62.2335000, -6.5930000),   
    'DKSTN': (55.2003000, 11.2572000),   
    'DKEND': (54.9087000, 9.6025000),   
    'DKHIR': (57.5935000, 9.9685000),   
    'DKBDX': (54.8381700, 11.4913500), 
    'DKANH': (56.7142000, 11.5071000),  
    'GHTEM': (5.6328500, 0.0127000),     
    'PLPLC': (53.5625200, 14.5888100),   
    'FIHKO': (59.8160000, 22.9360000),  
    'DEHHF': (54.3745000, 10.9990000),   
    'DKKRA': (54.9161125, 11.3605820),   
    'DKASN': (55.2685000, 9.8875010),    
}

coords = enriched_reference['LOCODE'].map(manual_coords)
lookup = coords.notnull()

enriched_reference.loc[lookup, 'Latitude'] = coords[lookup].str[0]
enriched_reference.loc[lookup, 'Longitude'] = coords[lookup].str[1]

enriched_nnull = enriched_reference[enriched_reference['Latitude'].notnull() & enriched_reference['Longitude'].notnull()]
print(layer_2_reference['mapped_LOCODE'].isin(enriched_nnull['LOCODE']).sum())
print(layer_2_reference['mapped_LOCODE'].count())

All layer 2 destinations have coordinates. We can now move on to layer 3: Route-like strings where we'll match the destinations to LOCODES where the route is unambiguous.

In [ ]:
unmatched_dests = dest_stats[~dest_stats.index.isin(enriched_reference['LOCODE'])]
layer_3 = df_target[(df_target['Destination'].isin(unmatched_dests.index)) & (df_target['Destination'].str.contains(r'[<>=/-]',na=False)) & (~df_target['Destination'].isin(layer_2_reference.index))]
layer_3_stats = layer_3.groupby('Destination').agg(observations=('MMSI', 'size'), unique_vessels=('MMSI', 'nunique'))
print(layer_3_stats.sort_values('observations', ascending=False).head(60))

We have 145 unique destination strings. Throughout these we can easily notice some patterns. First and most important for us are destination strings with a single directional arrow indicating a single origin and destination. The right-hand LOCODE in each string is a candidate for a valid destination LOCODE. Coming in second are destination strings with bidirectional arrows like <> <-> <=> and -. These could indicate ferry two-way routes. Finally we have ambiguous destination strings like PU-RO and AGERSO-STIGS-OMOE which contain abbreviations or multiple intermediate stops. Lets start off with the easiest ones being single direction destination strings. We will isolate strings containing a one-way > pattern, extract the right-hand candidate LOCODE, and retain only those whose candidate exists in our coordinate-ready enriched_reference.

In [ ]:
directional = layer_3_stats[(layer_3_stats.index.str.contains(r'[>]', na=False)) & (~layer_3_stats.index.str.contains(r'[<]', na=False))].copy()
directional['candidate_dest'] = directional.index.str.split('>').str[-1]
print(directional['candidate_dest'].nunique())
print(directional['candidate_dest'].isin(enriched_reference['LOCODE']).sum())
print(directional.shape)
directional = directional[directional['candidate_dest'].isin(enriched_reference['LOCODE']) & directional['candidate_dest'].isin(enriched_nnull['LOCODE'])].copy()
print(directional.shape)
print(directional['candidate_dest'].nunique())

directional['mapped_LOCODE'] = directional['candidate_dest']
directional['class'] = 'directional_route'

After filtering to directional route strings whose right-hand destination matches a valid LOCODE in enriched_reference, 43 route strings remain, representing 29 unique candidate destination LOCODEs. Next, we'll examine bidirectional and multi-stop route strings, which cannot be mapped using the same right-hand-endpoint criteria.

In [ ]:
bi_directional = layer_3_stats[(layer_3_stats.index.str.contains(r'[>]', na=False)) & (layer_3_stats.index.str.contains(r'[<]', na=False))].copy()
print(bi_directional)

 Bi directional strings do not act the same as single direction ones. They do not tell us which endpoint is the current destination, so if we mapped mapped_LOCODE = right_hand string we'd possibly be wrong. We could determine the current active destination however that would add additional complexity to layer 3 which doesnt really yield same value results. Concluding we will be only keeping directional strings for layer 3

In [ ]:
layer_3_reference = directional.copy()
layer_3_reference = layer_3_reference[['observations', 'unique_vessels', 'class', 'mapped_LOCODE']]
layer_1_reference = dest_stats[dest_stats.index.isin(enriched_nnull['LOCODE'])].copy()
layer_1_reference['class'] = 'exact_locode'
layer_1_reference['mapped_LOCODE'] = layer_1_reference.index
print(layer_1_reference)

Now all 3 layer reference datasets follow the same schema. We can go ahead and concatenate the three layers into the final port lookup table. The final table should have 382 rows and 4 columns.

In [ ]:
final_reference = pd.concat([layer_1_reference, layer_2_reference, layer_3_reference])
print(final_reference.shape)
print(final_reference.index.duplicated().sum())
print(final_reference['mapped_LOCODE'].isin(enriched_nnull['LOCODE']).sum())
print(final_reference['mapped_LOCODE'].nunique())

Our final port lookup table is working exactly as intended. Our next and final goal is to apply it to df target so each ais observation gets its resolved mapped_LOCODE and class (where available) aswell as each LOCODES coordinates.

In [ ]:
df_target.loc[df_target['Destination'].isin(final_reference.index), 'mapped_LOCODE'] = df_target['Destination'].map(final_reference['mapped_LOCODE'])
df_target.loc[df_target['Destination'].isin(final_reference.index), 'class'] = df_target['Destination'].map(final_reference['class'])

final_coverage = df_target['mapped_LOCODE'].notnull().sum()
final_vessels = df_target['MMSI'][df_target['mapped_LOCODE'].notnull()].nunique()

print(f"{((final_coverage/df_target.shape[0])*100):.2f}%")
print(f"{((final_vessels/df_target['MMSI'].nunique())*100):.2f}%")

enriched_nnull = enriched_nnull[enriched_nnull['LOCODE'].notnull()]

lat_index = enriched_nnull.set_index('LOCODE')['Latitude']
lon_index = enriched_nnull.set_index('LOCODE')['Longitude']

df_target['dest_lat'] = df_target['mapped_LOCODE'].map(lat_index)
df_target['dest_lon'] = df_target['mapped_LOCODE'].map(lon_index)

final_coord_coverage = (df_target['dest_lat'].notnull() & df_target['dest_lon'].notnull()).sum()
print(final_coord_coverage)


Destination mapping has been completed giving us a 74.85% observation coverage and a 73.59% vessel coverage

In [78]:
df_target.to_csv('../outputs/ais_mapped.csv')